# Decoding Customer Value: A SQL-driven Retention Strategy
from Summer Projects 2026 Consulting & Analytics Club IIT Guwahati

## Objective

This project prepares customer transaction data for segmentation analysis by:

- Cleaning and standardizing raw data
- Optimizing data types
- Engineering customer loyalty and value metrics
- Preparing a final analytics-ready dataset

The resulting dataset is designed for SQL-based customer segmentation and business intelligence analysis.

## Import Libraries and Load Dataset

Load required libraries and import the raw dataset.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv("Dataset.csv")

df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


## Data Cleaning

Standardize column names, handle missing values, and ensure consistent formatting across the dataset.

### Standardize Column Names

Convert column names to a consistent snake_case format for easier analysis and SQL compatibility.

In [2]:
df.rename(columns={"Purchase Amount (USD)": "purchase_amount"}, inplace=True)

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

### Handle Missing Values

Missing review ratings are replaced with the median rating of the corresponding product category.

In [3]:
df["review_rating"] = (
    df.groupby("category")["review_rating"]
      .transform(lambda x: x.fillna(x.median()))
)

## Data Type Optimization

Convert columns to appropriate data types to reduce memory usage and improve analytical efficiency.

In [4]:
#categorical features
cat_cols = [
    'gender','category','location','size',
    'color','season','shipping_type',
    'payment_method','item_purchased'
]

df[cat_cols] = df[cat_cols].astype("category")

#Boolean features
df["promo_code_used"] = (
    df["promo_code_used"]
      .map({"Yes": True, "No": False})
      .astype(bool)
)

df["discount_applied"] = (
    df["discount_applied"]
      .map({"Yes": True, "No": False})
      .astype(bool)
)

df["subscription_status"] = (
    df["subscription_status"]
      .map({"Yes": True, "No": False})
      .astype(bool)
)

#Numeric Features
df["age"] = df["age"].astype("int8")
df["purchase_amount"] = df["purchase_amount"].astype("int32")
df["review_rating"] = df["review_rating"].astype("float32")
df["previous_purchases"] = df["previous_purchases"].astype("int32")

## Ordinal Feature Encoding

Variables which contain a natural ranking are stored as ordered categories.

In [ ]:
#Purchase Frequency
df["frequency_of_purchases"] = df["frequency_of_purchases"].replace({
    "Bi-Weekly": "Fortnightly",
    "Every 3 Months": "Quarterly"
})

purchase_order = [
    "Annually",
    "Quarterly",
    "Monthly",
    "Fortnightly",
    "Weekly"
]

df["frequency_of_purchases"] = pd.Categorical(
    df["frequency_of_purchases"],
    categories=purchase_order,
    ordered=True
)

#Product Size
size_order = ["S", "M", "L", "XL"]

df["size"] = pd.Categorical(
    df["size"],
    categories=size_order,
    ordered=True
)

## Feature Engineering

Create business-focused metrics that measure customer value, loyalty, sentiment, and revenue quality.

### Estimated Lifetime Value (eLTV)

Approximates customer value by combining purchase amount and purchase history.

In [6]:
df["e_LTV"] = (
    df["purchase_amount"] *
    (df["previous_purchases"] + 1)
)

### Customer Value Tier

Segments customers into value-based groups using eLTV distribution percentiles.

In [7]:
quantiles = [0, 0.30, 0.70, 0.90, 1.0]

tier_labels = [
    "Bronze",
    "Silver",
    "Gold",
    "Platinum"
]

df["customer_value_tier"] = pd.qcut(
    df["e_LTV"],
    q=quantiles,
    labels=tier_labels
)

### Sentiment Health Flag

Classifies customers based on review ratings to estimate brand sentiment.

In [8]:
bins = [0, 3.499, 4.499, np.inf]

sentiment_labels = [
    "Detractor",
    "Passive Buyer",
    "Brand Promoter"
]

df["sentiment_health_flag"] = pd.cut(
    df["review_rating"],
    bins=bins,
    labels=sentiment_labels,
    include_lowest=True
)

### Promo Dependency Index (PDI)

Measures dependence on discounts and promotional offers.

In [9]:
df["pdi"] = (
    df["discount_applied"].astype(int) +
    df["promo_code_used"].astype(int)
) / 2

### Feature Normalization

Normalize variables before combining them into composite loyalty metrics.

In [10]:
freq_map = {
    "Weekly": 5,
    "Fortnightly": 4,
    "Monthly": 3,
    "Quarterly": 2,
    "Annually": 1
}

df["freq_score"] = (
    df["frequency_of_purchases"]
      .astype(str)
      .map(freq_map)
)

scaler = MinMaxScaler()

cols_to_normalize = [
    "previous_purchases",
    "purchase_amount",
    "freq_score"
]

normalized = scaler.fit_transform(
    df[cols_to_normalize]
)

df[[
    "prev_purchases_norm",
    "purchase_amount_norm",
    "freq_score_norm"
]] = normalized

### Behavioural Loyalty Score (BLS)

Measures loyalty through repeat purchasing behavior and purchase frequency.

In [11]:
df["BLS"] = (
    0.7 * df["prev_purchases_norm"] +
    0.3 * df["freq_score_norm"]
)

### Economic Loyalty Score (ELS)

Measures financial value while accounting for subscription status and promotional dependence.

In [12]:
df["ELS"] = (
    0.5 * df["purchase_amount_norm"] +
    0.3 * df["subscription_status"].astype(float) +
    0.2 * (1 - df["pdi"])
)

### Revenue Quality Index (RQI)

Evaluates revenue quality by combining spending, satisfaction, and promotional independence.

In [13]:
df["review_norm"] = (
    df["review_rating"] - 1
) / 4

df["RQI"] = (
    0.5 * df["purchase_amount_norm"] +
    0.3 * df["review_norm"] +
    0.2 * (1 - df["pdi"])
)

## Validation

Review the final dataset structure and verify engineered metrics.

In [14]:
df.info()

df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 30 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   customer_id             3900 non-null   int64   
 1   age                     3900 non-null   int8    
 2   gender                  3900 non-null   category
 3   item_purchased          3900 non-null   category
 4   category                3900 non-null   category
 5   purchase_amount         3900 non-null   int32   
 6   location                3900 non-null   category
 7   size                    3900 non-null   category
 8   color                   3900 non-null   category
 9   season                  3900 non-null   category
 10  review_rating           3900 non-null   float32 
 11  subscription_status     3900 non-null   bool    
 12  shipping_type           3900 non-null   category
 13  discount_applied        3900 non-null   bool    
 14  promo_code_used         

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,...,sentiment_health_flag,pdi,freq_score,prev_purchases_norm,purchase_amount_norm,freq_score_norm,BLS,ELS,review_norm,RQI
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,...,Detractor,1.0,4,0.265306,0.4125,0.75,0.410714,0.50625,0.525,0.36375
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,...,Detractor,1.0,4,0.020408,0.5500,0.75,0.239286,0.57500,0.525,0.43250
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,...,Detractor,1.0,5,0.448980,0.6625,1.00,0.614286,0.63125,0.525,0.48875
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,...,Passive Buyer,1.0,5,0.979592,0.8750,1.00,0.985714,0.73750,0.625,0.62500
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,...,Detractor,1.0,1,0.612245,0.3625,0.00,0.428571,0.48125,0.425,0.30875


## Export Processed Dataset

Save the cleaned and engineered dataset for SQL analysis and dashboarding.

In [15]:
df.to_csv(
    "Cleaned_Dataset.csv",
    index=False
)

df.to_csv(
    "Cleaned_Dataset.xlsx",
    index=False
)